# <span style='color: pink'>**FINAL CSV DATASETS**</span>

## <span style='color: white'>**Final Matches Datasets**</span>

### **1. Joining all the files**

#### 1.1 Import libraries

In [120]:
import pandas as pd
from pathlib import Path

#### 1.2 Define the correct folder paths

In [121]:
ATP_DIR = Path("data/github_raw_data/ATP_matches_2000-2024")
WTA_DIR = Path("data/github_raw_data/WTA_matches_2000-2024")

#### 1.3 Check available files

In [122]:
YEARS = list(range(2000, 2025))

def expected_files(prefix, folder):
    files = []
    missing = []
    
    for year in YEARS:
        file_path = folder / f"{prefix}_matches_{year}.csv"
        if file_path.exists():
            files.append(file_path)
        else:
            missing.append(file_path.name)
            
    return files, missing

atp_files, atp_missing = expected_files("atp", ATP_DIR)
wta_files, wta_missing = expected_files("wta", WTA_DIR)

print("ATP files found:", len(atp_files))
print("ATP missing:", atp_missing)

print("WTA files found:", len(wta_files))
print("WTA missing:", wta_missing)


ATP files found: 25
ATP missing: []
WTA files found: 25
WTA missing: []


#### 1.4 Merge all files

1.4.1 ATP files

In [123]:
def load_and_concat(files):
    dfs = []
    
    for f in sorted(files):
        year = int(f.stem.split("_")[-1])
        df = pd.read_csv(f, low_memory=False)
        df["source_year"] = year
        dfs.append(df)
    
    big_df = pd.concat(dfs, ignore_index=True, sort=False)
    return big_df

atp_big = load_and_concat(atp_files)

print("ATP dataset shape:", atp_big.shape)


ATP dataset shape: (74906, 50)


1.4.2 WTA files

In [124]:
wta_big = load_and_concat(wta_files)

print("WTA dataset shape:", wta_big.shape)


WTA dataset shape: (68624, 50)


#### 1.5 Quick sanity check

In [125]:
print("ATP years:", atp_big["source_year"].min(), "→", atp_big["source_year"].max())
print("WTA years:", wta_big["source_year"].min(), "→", wta_big["source_year"].max())


ATP years: 2000 → 2024
WTA years: 2000 → 2024


#### 1.6 Save the complete datasets

In [126]:
atp_big.to_csv("data/github_raw_data/ATP_matches_2000_2024_full.csv", index=False)
wta_big.to_csv("data/github_raw_data/WTA_matches_2000_2024_full.csv", index=False)

print("Files saved successfully.")


Files saved successfully.


### **2. Data Cleaning and Preprocessing**

#### 2.1 Load the full datasets

ATP dataset:

In [127]:
import pandas as pd
import numpy as np

atp_full_dataset = pd.read_csv("data/github_raw_data/ATP_matches_2000_2024_full.csv", low_memory=False)
atp_full_dataset.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,source_year
0,2000-301,Auckland,Hard,32,A,20000110,1,103163,1.0,NaN,...,39.0,29.0,17.0,4.0,7.0,11.0,1612.0,63.0,595.0,2000
1,2000-301,Auckland,Hard,32,A,20000110,2,102607,NaN,Q,...,25.0,18.0,12.0,3.0,6.0,211.0,157.0,49.0,723.0,2000
2,2000-301,Auckland,Hard,32,A,20000110,3,103252,NaN,NaN,...,20.0,7.0,8.0,7.0,11.0,48.0,726.0,59.0,649.0,2000
3,2000-301,Auckland,Hard,32,A,20000110,4,103507,7.0,NaN,...,29.0,14.0,10.0,6.0,8.0,45.0,768.0,61.0,616.0,2000
4,2000-301,Auckland,Hard,32,A,20000110,5,102103,NaN,Q,...,34.0,18.0,12.0,5.0,9.0,167.0,219.0,34.0,873.0,2000


WTA dataset:

In [128]:
wta_full_dataset = pd.read_csv("data/github_raw_data/WTA_matches_2000_2024_full.csv", low_memory=False)
wta_full_dataset.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,source_year
0,2000-D001,Fed Cup G1 PO: JPN vs CHN,Hard,4,D,20000430,1,201419,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,190.0,121.0,125.0,205.0,2000
1,2000-D001,Fed Cup G1 PO: JPN vs CHN,Hard,4,D,20000430,2,200085,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,22.0,1230.0,96.0,319.0,2000
2,2000-D002,Fed Cup WG SF: USA vs BEL,Carpet,4,D,20001122,1,200652,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,4.0,3255.0,48.0,661.0,2000
3,2000-D002,Fed Cup WG SF: USA vs BEL,Carpet,4,D,20001122,2,200128,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.0,5022.0,18.0,1398.0,2000
4,2000-D003,Fed Cup WG SF: CZE vs ESP,Carpet,4,D,20001121,1,200017,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,9.0,2132.0,54.0,610.0,2000


#### **Dataset Description**

The `matches` datasets provide detailed match-level information for ATP and WTA tournaments.  
Each row corresponds to a single match and includes tournament characteristics, player information, match outcomes, and performance statistics.

Below is a structured description of the key variables.

---

##### **1. Tournament Information**

**tourney_id**  
A unique tournament identifier (e.g., `2020-888`).  
The first four digits represent the year. The remaining characters depend on the data source and do not follow a fully standardized format.

**tourney_name**  
Official name of the tournament.

**surface**  
Playing surface of the event (Hard, Clay, Grass, Carpet).

**draw_size**  
Number of players in the main draw.  
Values are often rounded up to the nearest power of two (e.g., a 28-player draw may appear as 32).

**tourney_level**  
Indicates the category of the tournament.

ATP categories:
- `G` — Grand Slam  
- `M` — Masters 1000  
- `A` — Other tour-level events (ATP 500 / ATP 250)  
- `F` — Tour Finals and season-ending events  
- `D` — Davis Cup  
- `C` — Challenger  
- `S` — Satellite / ITF  

WTA categories:
- `P` — Premier  
- `PM` — Premier Mandatory  
- `I` — International  
- Numeric codes (e.g., `15`) — ITF tournaments classified by prize money  
- `T1`, `T2`, etc. — Historical Tier designations  
- `D` — Federation Cup / Fed Cup / Billie Jean King Cup  

Other codes (across tours):
- `E` — Exhibition  
- `J` — Junior tournaments  
- `T` — Team tennis events  

---

##### **2. Match Information**

**tourney_date**  
Tournament start date in `YYYYMMDD` format, typically corresponding to the Monday of the tournament week.

**match_num**  
Match-specific identifier within a tournament.  
Numbering schemes vary across tournaments and sources.

**round**  
Stage of the tournament (e.g., R32, QF, SF, F).

**best_of**  
Number of sets required to win the match:
- `3` — Best-of-three  
- `5` — Best-of-five  

**minutes**  
Match duration in minutes (when available).

**score**  
Final match score.

---

##### **3. Player Information**

Each match includes detailed information for both the winner and the loser.

##### **Winner Variables**

- **winner_id** — Unique player identifier within the dataset  
- **winner_seed** — Tournament seed  
- **winner_entry** — Entry status (WC, Q, LL, PR, ITF, etc.)  
- **winner_name** — Player’s full name  
- **winner_hand** — Playing hand (R = Right, L = Left, U = Unknown)  
- **winner_ht** — Height in centimeters (when available)  
- **winner_ioc** — Three-letter country code  
- **winner_age** — Age (in years) at the time of the tournament  
- **winner_rank** — Official ATP/WTA ranking at tournament start  
- **winner_rank_points** — Ranking points (when available)

##### **Loser Variables**

The following variables mirror those of the winner:

- loser_id  
- loser_seed  
- loser_entry  
- loser_name  
- loser_hand  
- loser_ht  
- loser_ioc  
- loser_age  
- loser_rank  
- loser_rank_points  

---

##### **4. Match Performance Statistics**

Performance statistics are recorded separately for winners (`w_` prefix) and losers (`l_` prefix).

##### **Winner Statistics** (`w_`)

- **w_ace** — Aces  
- **w_df** — Double faults  
- **w_svpt** — Serve points played  
- **w_1stIn** — First serves made  
- **w_1stWon** — First-serve points won  
- **w_2ndWon** — Second-serve points won  
- **w_SvGms** — Service games played  
- **w_bpSaved** — Break points saved  
- **w_bpFaced** — Break points faced  

##### **Loser Statistics** (`l_`)

- l_ace  
- l_df  
- l_svpt  
- l_1stIn  
- l_1stWon  
- l_2ndWon  
- l_SvGms  
- l_bpSaved  
- l_bpFaced  

#### 2.2 Keep only the relevant columns

To construct a clean graph representation of professional tennis competitions, only variables necessary to identify tournaments, matches, and players were retained from the match dataset. Detailed performance statistics and ranking information were excluded at this stage, as these variables are represented separately in dedicated ranking and player datasets. This normalization prevents redundancy and allows entities such as players, matches, tournaments, and rankings to be modeled as distinct nodes within the graph database.

ATP dataset:

In [129]:
keep_columns = [
    "tourney_id", "tourney_name", "tourney_date", "tourney_level", "surface", "draw_size",
    "match_num", "round", "score", "best_of", 
    "winner_id", "winner_name", 
    "loser_id", "loser_name",
    "source_year"
]

# keep only columns that actually exist 
existing_cols = [c for c in keep_columns if c in atp_full_dataset.columns]
atp = atp_full_dataset[existing_cols].copy()

print("ATP kept columns:", len(atp.columns))
print("ATP shape after column subset:", atp.shape)
atp.head()

ATP kept columns: 15
ATP shape after column subset: (74906, 15)


,tourney_id,tourney_name,tourney_date,tourney_level,surface,draw_size,match_num,round,score,best_of,winner_id,winner_name,loser_id,loser_name,source_year
0,2000-301,Auckland,20000110,A,Hard,32,1,R32,7-5 4-6 7-5,3,103163,Tommy Haas,101543,Jeff Tarango,2000
1,2000-301,Auckland,20000110,A,Hard,32,2,R32,7-5 7-5,3,102607,Juan Balcells,102644,Franco Squillari,2000
2,2000-301,Auckland,20000110,A,Hard,32,3,R32,6-3 6-1,3,103252,Alberto Martin,102238,Alberto Berasategui,2000
3,2000-301,Auckland,20000110,A,Hard,32,4,R32,6-4 6-4,3,103507,Juan Carlos Ferrero,103819,Roger Federer,2000
4,2000-301,Auckland,20000110,A,Hard,32,5,R32,0-6 7-6(7) 6-1,3,102103,Michael Sell,102765,Nicolas Escude,2000


WTA dataset:

In [130]:
keep_columns = [
    "tourney_id", "tourney_name", "tourney_date", "tourney_level", "surface", "draw_size",
    "match_num", "round", "score", "best_of", 
    "winner_id", "winner_name", 
    "loser_id", "loser_name",
    "source_year"
]

# keep only columns that actually exist 
existing_cols = [c for c in keep_columns if c in wta_full_dataset.columns]
wta = wta_full_dataset[existing_cols].copy()

print("WTA kept columns:", len(wta.columns))
print("WTA shape after column subset:", wta.shape)
wta.head()

WTA kept columns: 15
WTA shape after column subset: (68624, 15)


,tourney_id,tourney_name,tourney_date,tourney_level,surface,draw_size,match_num,round,score,best_of,winner_id,winner_name,loser_id,loser_name,source_year
0,2000-D001,Fed Cup G1 PO: JPN vs CHN,20000430,D,Hard,4,1,RR,7-6(6) 6-2,3,201419,Na Li,201085,Shinobu Asagoe,2000
1,2000-D001,Fed Cup G1 PO: JPN vs CHN,20000430,D,Hard,4,2,RR,6-4 6-2,3,200085,Ai Sugiyama,200075,Jing Qian Yi,2000
2,2000-D002,Fed Cup WG SF: USA vs BEL,20001122,D,Carpet,4,1,RR,7-6(1) 6-2,3,200652,Monica Seles,200003,Justine Henin,2000
3,2000-D002,Fed Cup WG SF: USA vs BEL,20001122,D,Carpet,4,2,RR,7-6(4) 4-6 6-3,3,200128,Lindsay Davenport,200079,Kim Clijsters,2000
4,2000-D003,Fed Cup WG SF: CZE vs ESP,20001121,D,Carpet,4,1,RR,5-7 6-4 6-3,3,200017,Arantxa Sanchez Vicario,201083,Daja Bedanova,2000


#### 2.3 Filter tournament level
Only tournaments belonging to the main ATP and WTA tours were retained. Team competitions (e.g., Davis Cup and Fed Cup), lower-level circuits (e.g., Challenger), and irregular tournament categories (e.g. exhibitions and junior circuits) were excluded in order to ensure that the network represents competitive interactions within the primary professional tours.

Removing minor tournaments helps because:
- the network focuses on elite professional competition
- players participating are comparable across tours
- the network becomes less noisy

In [131]:
print(atp["tourney_level"].value_counts())
print(wta["tourney_level"].value_counts())

tourney_level
A    40847
M    13998
G    12573
D     6987
F      437
O       64
Name: count, dtype: int64
tourney_level
I     14818
G     12573
P     10491
D      7055
T1     4651
T3     4562
T2     4329
PM     4303
T4     2912
T5      913
CC      756
F       494
O       446
W       321
Name: count, dtype: int64


(WTA list is larger because the WTA circuit changed format over time.)

In [132]:
atp = atp[atp["tourney_level"].isin(["G", "M", "A", "F"])] #G - Grand Slam, M - Masters 1000, A - ATP 250 / 500, F - Tour Finals
wta = wta[wta["tourney_level"].isin(["G", "P", "PM", "I", "T1", "T2", "T3", "T4", "T5", "W", "F"])] #G - Grand Slam, P - Premier, PM - Premier Mandatory, I - International, T - Tier Tournaments, F - Finals

In [133]:
print("ATP rows:", len(atp))
print("WTA rows:", len(wta))

ATP rows: 67855
WTA rows: 60367


#### 2.4 Normalize string columns

It standardizes text fields and avoids problems such as extra spaces or inconsistent strings that could later generate duplicate nodes in Neo4j.

In [134]:
def clean_str(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .str.normalize("NFKC")
         .str.strip()
         .str.replace(r"\s+", " ", regex=True)
    )

text_cols = [
    "tourney_id", "tourney_name", "tourney_level", "surface",
    "round", "score", "winner_name", "loser_name"
]

for c in text_cols:
    atp[c] = clean_str(atp[c])
    wta[c] = clean_str(wta[c])

#### 2.5 Enforce numeric columns

In [135]:
numeric_cols = ["winner_id", "loser_id", "best_of", "match_num", "draw_size", "source_year"]

for c in numeric_cols:
    atp[c] = pd.to_numeric(atp[c], errors="coerce").astype("Int64")
    wta[c] = pd.to_numeric(wta[c], errors="coerce").astype("Int64")

#### 2.6 Convert date
It converts the date into a real datetime format, which is more suitable for filtering, validation, and Neo4j date properties.

In [136]:
atp["tourney_date"] = pd.to_datetime(
    atp["tourney_date"], format="%Y%m%d", errors="coerce"
)

wta["tourney_date"] = pd.to_datetime(
    wta["tourney_date"], format="%Y%m%d", errors="coerce"
)

#### 2.7 Drop rows that break integrity
This removes rows that cannot be reliably modeled as tennis matches, such as:
- matches with missing essential attributes
- corrupted rows where winner and loser are the same player

In [137]:
required = [
    "tourney_id", "tourney_name", "tourney_date", "tourney_level", "surface", "draw_size",
    "match_num", "round", "score", "best_of",
    "winner_id", "winner_name",
    "loser_id", "loser_name",
    "source_year"
]

before = len(atp)
atp = atp.dropna(subset=required)
atp = atp[atp["winner_id"] != atp["loser_id"]]
print(f"Rows removed by integrity rules in ATP: {before - len(atp)}")
print("Remaining ATP rows:", len(atp))

before = len(wta)
wta = wta.dropna(subset=required)
wta = wta[wta["winner_id"] != wta["loser_id"]]
print(f"Rows removed by integrity rules in WTA: {before - len(wta)}")
print("Remaining WTA rows:", len(wta))

Rows removed by integrity rules in ATP: 0
Remaining ATP rows: 67855
Rows removed by integrity rules in WTA: 0
Remaining WTA rows: 60367


#### 2.8 Remove exact duplicates and Walkovers
It prevents duplicated match records from being imported into the graph and creating repeated match nodes or incorrect edge counts.

In [138]:
before = len(atp)
atp = atp.drop_duplicates(subset=required, keep="first")
print(f"Exact duplicates removed in ATP dataset: {before - len(atp)}")

before = len(wta)
wta = wta.drop_duplicates(subset=required, keep="first")
print(f"Exact duplicates removed in WTA dataset: {before - len(wta)}")

Exact duplicates removed in ATP dataset: 0
Exact duplicates removed in WTA dataset: 0


In [139]:
# Remove ATP walkovers 
before = len(atp)
atp = atp[~atp['score'].str.contains(
    r'W/O|walkover|def\.', case=False, na=False, regex=True
)]
print(f"Removed walkovers: {before - len(atp)}")

# Remove WTA walkovers 
before = len(wta)
wta = wta[~wta['score'].str.contains(
    r'W/O|walkover|def\.', case=False, na=False, regex=True
)]
print(f"Removed walkovers: {before - len(wta)}")

Removed walkovers: 367
Removed walkovers: 364


#### 2.9 Check for missing values

In [140]:
def inspect_missing(df, name="Dataset"):
    print(f"\n{name}")
    print("="*50)
    
    total_rows = len(df)
    
    missing_count = df.isna().sum()
    missing_percent = (missing_count / total_rows) * 100
    
    summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent.round(2)
    })
    
    summary = summary[summary["missing_count"] > 0].sort_values("missing_percent", ascending=False)
    
    if summary.empty:
        print("No missing values detected.")
    else:
        print(summary)

inspect_missing(atp, "ATP Dataset")
inspect_missing(wta, "WTA Dataset")


ATP Dataset
No missing values detected.

WTA Dataset
No missing values detected.


### **3. Graph Data Modeling and Neo4j Preparation**

#### 3.1 Merge ATP and WTA into a unique matches dataset

In [141]:
import pandas as pd

atp = atp.copy()
wta = wta.copy()

atp["tour"] = "ATP"
wta["tour"] = "WTA"

matches = pd.concat([atp, wta], ignore_index=True)
print("Total matches after ATP+WTA merge:", len(matches))

Total matches after ATP+WTA merge: 127491


#### 3.2 Create global identifiers (keys) for Neo4j

In [142]:
# Ensure numeric fields are clean and nullable-safe
for c in ["winner_id", "loser_id", "best_of", "match_num", "draw_size", "source_year"]:
    matches[c] = pd.to_numeric(matches[c], errors="coerce").astype("Int64")

# Create globally unique player keys (avoid ATP/WTA collisions)
matches["player_winner_key"] = matches["tour"] + "_" + matches["winner_id"].astype("string")
matches["player_loser_key"]  = matches["tour"] + "_" + matches["loser_id"].astype("string")

# Create globally unique tournament key
matches["tourney_key"] = matches["tour"] + "_" + matches["tourney_id"].astype("string")

#### 3.3 Create match_id (primary key)

In [143]:
matches["match_id"] = (
    matches["tour"] + "_"
    + matches["tourney_id"].astype("string") + "_"
    + matches["source_year"].astype("string") + "_" #to prevent cross-year collisions
    + matches["match_num"].astype("string")
)

#### 3.4 Graph integrity checks (Neo4j-safe)

In [144]:
# Fix 1: Rebuild match_id including round to prevent collisions
# from tournaments with RR + knockout phases where match_num resets
matches["match_id"] = (
    matches["tour"] + "_"
    + matches["tourney_id"].astype("string") + "_"
    + matches["source_year"].astype("string") + "_"
    + matches["round"].astype("string") + "_"
    + matches["match_num"].astype("string")
)

# Fix 2: Remove walkovers (W/O) — matches never played on court
# Retirements (RET) are kept as a winner was decided
before = len(matches)
matches = matches[~matches["score"].str.contains(
    r"W/O", case=False, na=False, regex=False
)]
print(f"Walkovers removed: {before - len(matches)}")
print(f"Remaining after walkover removal: {len(matches)}")

# Graph integrity checks
required_graph = [
    "match_id",
    "tourney_key", "tourney_id", "tourney_name", "tourney_date",
    "tourney_level", "surface", "draw_size",
    "match_num", "round", "score", "source_year",
    "player_winner_key", "player_loser_key"
]

before = len(matches)
matches = matches.dropna(subset=required_graph)
matches = matches[matches["player_winner_key"] != matches["player_loser_key"]]
print("Removed invalid matches:", before - len(matches))
print("Remaining matches:", len(matches))

# Diagnose duplicate match_ids
dupes = matches[matches.duplicated(subset=["match_id"], keep=False)]
print(f"Duplicate match_id rows: {len(dupes)}")
if len(dupes) > 0:
    print(dupes[[
        "match_id", "tourney_key", "tourney_date", "round",
        "score", "player_winner_key", "player_loser_key"
    ]].to_string())

# Remove any remaining duplicates
before = len(matches)
matches = matches.drop_duplicates(subset=["match_id"], keep="first")
print("Duplicate match_id removed:", before - len(matches))
print("Final match count:", len(matches))

Walkovers removed: 0
Remaining after walkover removal: 127491
Removed invalid matches: 0
Remaining matches: 127491
Duplicate match_id rows: 0
Duplicate match_id removed: 0
Final match count: 127491


#### 3.5 Export Neo4j-ready CSV files: tournaments + matches

In [145]:
OUT_TOURNEYS = "data/tournaments_all.csv"
OUT_MATCHES  = "data/matches_all.csv"

# Tournament nodes
tournaments = matches[[
    "tourney_id",
    "tourney_key",
    "tourney_name",
    "tourney_level",
    "surface",
    "draw_size",
    "tour"
]].drop_duplicates(subset=["tourney_key"]).copy()

tournaments.to_csv(OUT_TOURNEYS, index=False)
print("Saved tournaments:", len(tournaments), "->", OUT_TOURNEYS)

# Match nodes
matches_out = matches[[
    "match_id",
    "tourney_key",
    "tourney_date",
    "round",
    "score",
    "player_winner_key",
    "player_loser_key",
    "tour"
]].drop_duplicates(subset=["match_id"]).copy()

# Neo4j import-friendly date format (ISO)
matches_out["tourney_date"] = matches_out["tourney_date"].dt.date.astype("string")

matches_out.to_csv(OUT_MATCHES, index=False)
print("Saved matches:", len(matches_out), "->", OUT_MATCHES)

Saved tournaments: 3084 -> data/tournaments_all.csv
Saved matches: 127491 -> data/matches_all.csv


***
## <span style='color: white'>**Final Ranking Datasets**</span>

### **1. Joining all the files**

#### STEP 1: Import Libraries

In [146]:
import pandas as pd
from pathlib import Path
import re

#### STEP 2: Define the correct folder paths

In [147]:
ATP_RANK_DIR = Path("data/github_raw_data/atp_rankings_2000-2024")
WTA_RANK_DIR = Path("data/github_raw_data/wta_rankings_2000-2024") 

#### STEP 3: Check available files

In [148]:
def find_ranking_files(folder: Path):
    files = sorted([f for f in folder.glob("*.csv") 
                    if re.search(r"rank", f.name, flags=re.IGNORECASE)])
    return files

atp_rank_files = find_ranking_files(ATP_RANK_DIR)
wta_rank_files = find_ranking_files(WTA_RANK_DIR)

print("ATP ranking files:", [f.name for f in atp_rank_files])
print("WTA ranking files:", [f.name for f in wta_rank_files])

ATP ranking files: ['atp_rankings_00s.csv', 'atp_rankings_10s.csv', 'atp_rankings_20s.csv', 'atp_rankings_current.csv']
WTA ranking files: ['wta_rankings_00s.csv', 'wta_rankings_10s.csv', 'wta_rankings_20s.csv', 'wta_rankings_current.csv']


#### STEP 4: Merge all files

4.1 ATP files

In [149]:
def load_and_concat(files):
    dfs = []
    for f in files:
        df = pd.read_csv(f, low_memory=False)
        df["source_file"] = f.name
        dfs.append(df)
    big_df = pd.concat(dfs, ignore_index=True, sort=False)
    return big_df

atp_rank_big = load_and_concat(atp_rank_files)
print("ATP rankings shape:", atp_rank_big.shape)

ATP rankings shape: (2261808, 5)


4.2 WTA files

In [150]:
wta_rank_big = load_and_concat(wta_rank_files)
print("WTA rankings shape:", wta_rank_big.shape)

WTA rankings shape: (1543106, 6)


#### STEP 5: Quick sanity checks

In [151]:
def add_date_and_filter(df, date_col="ranking_date"):
    if date_col not in df.columns:
        raise ValueError(f"Column '{date_col}' not found. Available columns: {df.columns.tolist()}")

    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col].astype(str), format="%Y%m%d", errors="coerce")
    df = df[df[date_col].between("2000-01-01", "2024-12-31")]
    return df

atp_rank_big = add_date_and_filter(atp_rank_big, "ranking_date")
wta_rank_big = add_date_and_filter(wta_rank_big, "ranking_date")

print("ATP ranking dates:", atp_rank_big["ranking_date"].min(), "→", atp_rank_big["ranking_date"].max())
print("WTA ranking dates:", wta_rank_big["ranking_date"].min(), "→", wta_rank_big["ranking_date"].max())

ATP ranking dates: 2000-01-10 00:00:00 → 2024-12-30 00:00:00
WTA ranking dates: 2000-01-01 00:00:00 → 2024-12-30 00:00:00


#### STEP 6: Save the complete datasets

In [152]:
atp_rank_big.to_csv("data/github_raw_data/ATP_rankings_2000_2024_full.csv", index=False)
wta_rank_big.to_csv("data/github_raw_data/WTA_rankings_2000_2024_full.csv", index=False)

print("Ranking datasets saved successfully.")

Ranking datasets saved successfully.


### **2. Data Cleaning and Preprocessing**

#### 2.1 Load the full datasets

In [153]:
import pandas as pd
import numpy as np

ATP_RANKINGS_PATH = "data/github_raw_data/ATP_rankings_2000_2024_full.csv"
WTA_RANKINGS_PATH = "data/github_raw_data/WTA_rankings_2000_2024_full.csv"

atp_r_full = pd.read_csv(ATP_RANKINGS_PATH, low_memory=False)
wta_r_full = pd.read_csv(WTA_RANKINGS_PATH, low_memory=False)

print("ATP rankings shape:", atp_r_full.shape)
print("WTA rankings shape:", wta_r_full.shape)

display(atp_r_full.head())
display(wta_r_full.head())

ATP rankings shape: (2261808, 5)
WTA rankings shape: (1543106, 6)


,ranking_date,rank,player,points,source_file
0,2000-01-10,1,101736,4135.0,atp_rankings_00s.csv
1,2000-01-10,2,102338,2915.0,atp_rankings_00s.csv
2,2000-01-10,3,101948,2419.0,atp_rankings_00s.csv
3,2000-01-10,4,103017,2184.0,atp_rankings_00s.csv
4,2000-01-10,5,102856,2169.0,atp_rankings_00s.csv


,ranking_date,rank,player,points,tours,source_file
0,2000-01-01,1,200001,6074.0,NaN,wta_rankings_00s.csv
1,2000-01-03,1,200001,6074.0,NaN,wta_rankings_00s.csv
2,2000-01-10,1,200001,6074.0,NaN,wta_rankings_00s.csv
3,2000-01-17,1,200001,6003.0,NaN,wta_rankings_00s.csv
4,2000-01-24,1,200001,6003.0,NaN,wta_rankings_00s.csv


#### 2.2 Standardize column names

In [154]:
# (files use 'player' not 'player_id')
atp_r_full = atp_r_full.rename(columns={"player": "player_id"})
wta_r_full = wta_r_full.rename(columns={"player": "player_id"})

#### 2.3 Keep only relevant columns

In [155]:
# (drop 'source_file' and WTA 'tours' because not needed)
keep_cols = ["ranking_date", "player_id", "rank", "points"]

atp_r = atp_r_full[keep_cols].copy()
wta_r = wta_r_full[keep_cols].copy()

print("ATP kept cols:", atp_r.columns.tolist())
print("WTA kept cols:", wta_r.columns.tolist())

ATP kept cols: ['ranking_date', 'player_id', 'rank', 'points']
WTA kept cols: ['ranking_date', 'player_id', 'rank', 'points']


#### 2.4 Convert data types

In [156]:
atp_r["ranking_date"] = pd.to_datetime(atp_r["ranking_date"], errors="coerce")
wta_r["ranking_date"] = pd.to_datetime(wta_r["ranking_date"], errors="coerce")

for c in ["player_id", "rank"]:
    atp_r[c] = pd.to_numeric(atp_r[c], errors="coerce").astype("Int64")
    wta_r[c] = pd.to_numeric(wta_r[c], errors="coerce").astype("Int64")

# points can be float (some missing) -> keep numeric
atp_r["points"] = pd.to_numeric(atp_r["points"], errors="coerce")
wta_r["points"] = pd.to_numeric(wta_r["points"], errors="coerce")

#### 2.5 Integrity filtering

In [157]:
required = ["ranking_date", "player_id", "rank"]

before = len(atp_r)
atp_r = atp_r.dropna(subset=required)
atp_r = atp_r[atp_r["rank"] >= 1]
print("ATP removed by integrity:", before - len(atp_r), "| remaining:", len(atp_r))

before = len(wta_r)
wta_r = wta_r.dropna(subset=required)
wta_r = wta_r[wta_r["rank"] >= 1]
print("WTA removed by integrity:", before - len(wta_r), "| remaining:", len(wta_r))

ATP removed by integrity: 0 | remaining: 2261808
WTA removed by integrity: 0 | remaining: 1543106


#### 2.6 Resolve duplicates

In [158]:
# ATP: keep first after sorting by rank then points desc (stable)
atp_r = atp_r.sort_values(["player_id", "ranking_date", "rank", "points"], ascending=[True, True, True, False])
before = len(atp_r)
atp_r = atp_r.drop_duplicates(subset=["player_id", "ranking_date"], keep="first")
print("ATP duplicates resolved:", before - len(atp_r))

# WTA: keep best row (min rank; tie -> max points)
wta_r = wta_r.sort_values(["player_id", "ranking_date", "rank", "points"], ascending=[True, True, True, False])
before = len(wta_r)
wta_r = wta_r.drop_duplicates(subset=["player_id", "ranking_date"], keep="first")
print("WTA duplicates resolved:", before - len(wta_r))

ATP duplicates resolved: 613
WTA duplicates resolved: 190


#### 2.7 Check missing values

In [159]:
def inspect_missing(df, name="Dataset"):
    print(f"\n{name}")
    print("="*50)
    total_rows = len(df)
    missing_count = df.isna().sum()
    missing_percent = (missing_count / total_rows) * 100
    summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent.round(2)
    })
    summary = summary[summary["missing_count"] > 0].sort_values("missing_percent", ascending=False)
    if summary.empty:
        print("No missing values detected.")
    else:
        print(summary)

inspect_missing(atp_r, "ATP Rankings (cleaned)")
inspect_missing(wta_r, "WTA Rankings (cleaned)")


ATP Rankings (cleaned)
        missing_count  missing_percent
points            749             0.03

WTA Rankings (cleaned)
        missing_count  missing_percent
points              4              0.0


### **3. Graph Data Modeling and Neo4j Preparation**

#### 3.1 Merge ATP and WTA into a unique rankings dataset

In [160]:
import pandas as pd

atp_r = atp_r.copy()
wta_r = wta_r.copy()

atp_r["tour"] = "ATP"
wta_r["tour"] = "WTA"

rankings = pd.concat([atp_r, wta_r], ignore_index=True)
print("Total cleaned ranking rows:", len(rankings))

Total cleaned ranking rows: 3804111


#### 3.2 Create global identifiers (keys) for Neo4j

In [161]:
rankings["player_key"] = rankings["tour"] + "_" + rankings["player_id"].astype("string")

# ranking node key: one ranking per player per date
# Use ISO date string to keep it readable + stable
rankings["ranking_date_str"] = rankings["ranking_date"].dt.date.astype("string")
rankings["ranking_key"] = (
    rankings["tour"] + "_" +
    rankings["player_id"].astype("string") + "_" +
    rankings["ranking_date_str"]
)

#### 3.3 Graph integrity checks (Neo4j-safe)

In [162]:
required_graph = ["ranking_key", "player_key", "ranking_date", "rank"]

before = len(rankings)
rankings = rankings.dropna(subset=required_graph)
print("Removed invalid rows (graph rules):", before - len(rankings))
print("Remaining rows:", len(rankings))

# Ensure unique ranking nodes
before = len(rankings)
rankings = rankings.drop_duplicates(subset=["ranking_key"], keep="first")
print("Duplicate ranking_key removed:", before - len(rankings))

Removed invalid rows (graph rules): 0
Remaining rows: 3804111
Duplicate ranking_key removed: 0


#### 3.4 Export Neo4j-ready CSV files

In [163]:
OUT_RANKINGS = "data/rankings_all.csv"

rankings_out = rankings[[
    "ranking_key",
    "player_key",
    "ranking_date_str",
    "rank",
    "points",
    "tour"
]].rename(columns={"ranking_date_str": "ranking_date"}).copy()

rankings_out.to_csv(OUT_RANKINGS, index=False)
print("\nSaved rankings:", len(rankings_out))


Saved rankings: 3804111


***
## <span style='color: white'>**Final Players Dataset (ATP + WTA)**</span>

### **2. Data Cleaning and Preprocessing**

#### 2.1 Load the full datasets

In [164]:
import pandas as pd
import numpy as np

ATP_PLAYERS_PATH = "data/github_raw_data/ATP_players.csv"
WTA_PLAYERS_PATH = "data/github_raw_data/WTA_players.csv"

atp_p_full = pd.read_csv(ATP_PLAYERS_PATH, low_memory=False)
wta_p_full = pd.read_csv(WTA_PLAYERS_PATH, low_memory=False)

print("ATP players:", atp_p_full.shape)
print("WTA players:", wta_p_full.shape)

display(atp_p_full.head())
display(wta_p_full.head())

ATP players: (65989, 8)
WTA players: (70036, 8)


,player_id,name_first,name_last,hand,dob,ioc,height,wikidata_id
0,100001,Gardnar,Mulloy,R,19131122.0,USA,185.0,Q54544
1,100002,Pancho,Segura,R,19210620.0,ECU,168.0,Q54581
2,100003,Frank,Sedgman,R,19271002.0,AUS,180.0,Q962049
3,100004,Giuseppe,Merlo,R,19271011.0,ITA,NaN,Q1258752
4,100005,Richard,Gonzalez,R,19280509.0,USA,188.0,Q53554


,player_id,name_first,name_last,hand,dob,ioc,height,wikidata_id
0,113190,Bobby,Riggs,U,NaN,USA,NaN,NaN
1,200000,X,X,U,19000000.0,UNK,NaN,NaN
2,200001,Martina,Hingis,R,19800930.0,SUI,170.0,Q134720
3,200002,Mirjana,Lucic,R,19820309.0,CRO,181.0,Q239686
4,200003,Justine,Henin,R,19820601.0,BEL,167.0,Q11682


#### 2.2 Keep only relevant columns

In [165]:
keep_cols = ["player_id", "name_first", "name_last", "ioc"]

atp_p = atp_p_full[keep_cols].copy()
wta_p = wta_p_full[keep_cols].copy()

print("ATP kept columns:", atp_p.columns.tolist())
print("WTA kept columns:", wta_p.columns.tolist())

ATP kept columns: ['player_id', 'name_first', 'name_last', 'ioc']
WTA kept columns: ['player_id', 'name_first', 'name_last', 'ioc']


#### 2.3 Clean string columns

In [166]:
def clean_str(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .str.normalize("NFKC")
         .str.strip()
         .str.replace(r"\s+", " ", regex=True)
    )

for df in [atp_p, wta_p]:
    df["name_first"] = clean_str(df["name_first"])
    df["name_last"] = clean_str(df["name_last"])
    df["ioc"] = clean_str(df["ioc"])

#### 2.4 Create player_name column

In [167]:
for df in [atp_p, wta_p]:

    df["name_first"] = df["name_first"].fillna("")
    df["name_last"] = df["name_last"].fillna("")

    df["player_name"] = (
        df["name_first"].str.strip() + " " + df["name_last"].str.strip()
    ).str.strip()

    # Replace empty string with NaN (if both were missing)
    df["player_name"] = df["player_name"].replace("", pd.NA)

# Keep only final columns
atp_p = atp_p[["player_id", "player_name", "ioc"]]
wta_p = wta_p[["player_id", "player_name", "ioc"]]

#### 2.5 Convert player_id to numeric

In [168]:
for df in [atp_p, wta_p]:
    df["player_id"] = pd.to_numeric(df["player_id"], errors="coerce").astype("Int64")

#### 2.6 Integrity filtering

In [169]:
required = ["player_id", "player_name"]

before = len(atp_p)
atp_p = atp_p.dropna(subset=required)
print("ATP removed by integrity:", before - len(atp_p))

before = len(wta_p)
wta_p = wta_p.dropna(subset=required)
print("WTA removed by integrity:", before - len(wta_p))

# Remove duplicate player_id
atp_p = atp_p.drop_duplicates(subset=["player_id"])
wta_p = wta_p.drop_duplicates(subset=["player_id"])

print("ATP final players:", len(atp_p))
print("WTA final players:", len(wta_p))

ATP removed by integrity: 48
WTA removed by integrity: 0
ATP final players: 65941
WTA final players: 70036


In [170]:
def inspect_missing(df, name="Dataset"):
    print(f"\n{name}")
    print("="*50)
    total_rows = len(df)
    missing_count = df.isna().sum()
    missing_percent = (missing_count / total_rows) * 100
    summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent.round(2)
    })
    summary = summary[summary["missing_count"] > 0].sort_values("missing_percent", ascending=False)
    if summary.empty:
        print("No missing values detected.")
    else:
        print(summary)

inspect_missing(atp_p, "ATP Players (cleaned)")
inspect_missing(wta_p, "WTA Players (cleaned)")


ATP Players (cleaned)
     missing_count  missing_percent
ioc            664             1.01

WTA Players (cleaned)
     missing_count  missing_percent
ioc           1091             1.56


### **3. Graph Data Modeling and Neo4j Preparation**

#### 3.1 Merge ATP and WTA into a unique players dataset

In [171]:
import pandas as pd

atp_p = atp_p.copy()
wta_p = wta_p.copy()

atp_p["tour"] = "ATP"
wta_p["tour"] = "WTA"

players = pd.concat([atp_p, wta_p], ignore_index=True)
print("Total cleaned player rows:", len(players))

Total cleaned player rows: 135977


#### 3.2 Create global identifiers (keys) for Neo4j

In [172]:
players["player_key"] = players["tour"] + "_" + players["player_id"].astype("string")

#### 3.3 Graph integrity checks (Neo4j-safe)

In [173]:
players = players.dropna(subset=["player_key"])
players = players.drop_duplicates(subset=["player_key"]).reset_index(drop=True)

print("Duplicate player_key:", players["player_key"].duplicated().sum())

Duplicate player_key: 0


In [174]:
# Filter to only players who appear in at least one match
# Load matches_out if not still in memory
try:
    _ = matches_out
except NameError:
    matches_out = pd.read_csv("data/matches_all.csv", low_memory=False)

active_keys = set(matches_out["player_winner_key"]) | set(matches_out["player_loser_key"])

before = len(players)
players = players[players["player_key"].isin(active_keys)].copy().reset_index(drop=True)
print(f"Players filtered to active: {before} → {len(players)}")
# Expected: ~135,977 → ~2,700–3,000

Players filtered to active: 135977 → 3107


#### 3.4 Export Neo4j-ready CSV files

In [175]:
OUT_PLAYERS = "data/players_all.csv"

players_out = players[[
    "player_key",
    "player_id",
    "player_name",
    "ioc",
    "tour"
]].copy()

players_out.to_csv(OUT_PLAYERS, index=False)

print("Saved players:", len(players_out))

Saved players: 3107


***
## **FINAL CHECK**

In [176]:
import pandas as pd

PLAYERS = "data/players_all.csv"
TOURNEYS = "data/tournaments_all.csv"
MATCHES = "data/matches_all.csv"
RANKINGS = "data/rankings_all.csv"

players = pd.read_csv(PLAYERS, low_memory=False)
tourns  = pd.read_csv(TOURNEYS, low_memory=False)
matches = pd.read_csv(MATCHES, low_memory=False)
ranks   = pd.read_csv(RANKINGS, low_memory=False)

print("=== BASIC COUNTS ===")
print("Players:", len(players), "unique:", players["player_key"].nunique())
print("Tournaments:", len(tourns), "unique:", tourns["tourney_key"].nunique())
print("Matches:", len(matches), "unique:", matches["match_id"].nunique())
print("Rankings:", len(ranks), "unique:", ranks["ranking_key"].nunique())

print("\n=== TOUR SPLITS ===")
print("Players by tour:\n", players["tour"].value_counts())
print("Tournaments by tour:\n", tourns["tour"].value_counts())
print("Matches by tour:\n", matches["tour"].value_counts())
print("Rankings by tour:\n", ranks["tour"].value_counts())

print("\n=== KEY COVERAGE CHECKS ===")
players_set = set(players["player_key"])
tourns_set = set(tourns["tourney_key"])

missing_winners = set(matches["player_winner_key"]) - players_set
missing_losers  = set(matches["player_loser_key"]) - players_set
missing_rank_players = set(ranks["player_key"]) - players_set
missing_match_tourns = set(matches["tourney_key"]) - tourns_set

print("Missing winner keys in players:", len(missing_winners))
print("Missing loser keys in players:", len(missing_losers))
print("Missing ranking player keys in players:", len(missing_rank_players))
print("Missing tournament keys in tournaments:", len(missing_match_tourns))

if len(missing_winners) > 0:
    print("\nExample missing winner keys:", list(missing_winners)[:10])
if len(missing_losers) > 0:
    print("\nExample missing loser keys:", list(missing_losers)[:10])
if len(missing_rank_players) > 0:
    print("\nExample missing ranking player keys:", list(missing_rank_players)[:10])
if len(missing_match_tourns) > 0:
    print("\nExample missing tournament keys:", list(missing_match_tourns)[:10])

=== BASIC COUNTS ===
Players: 3107 unique: 3107
Tournaments: 3084 unique: 3084
Matches: 127491 unique: 127491
Rankings: 3804111 unique: 3804111

=== TOUR SPLITS ===
Players by tour:
 tour
ATP    1598
WTA    1509
Name: count, dtype: int64
Tournaments by tour:
 tour
ATP    1652
WTA    1432
Name: count, dtype: int64
Matches by tour:
 tour
ATP    67488
WTA    60003
Name: count, dtype: int64
Rankings by tour:
 tour
ATP    2261195
WTA    1542916
Name: count, dtype: int64

=== KEY COVERAGE CHECKS ===
Missing winner keys in players: 0
Missing loser keys in players: 0
Missing ranking player keys in players: 15967
Missing tournament keys in tournaments: 0

Example missing ranking player keys: ['ATP_104369', 'ATP_126528', 'WTA_209140', 'WTA_206086', 'ATP_122373', 'WTA_206209', 'ATP_106221', 'WTA_206149', 'ATP_201896', 'ATP_102697']


Ranking rows whose player identifiers were not present in the player registry were removed to guarantee referential integrity.

In [177]:
ranks_fixed = ranks[ranks["player_key"].isin(players["player_key"])].copy()
ranks_fixed.to_csv("data/rankings_all.csv", index=False)
print("Removed ranking rows:", len(ranks) - len(ranks_fixed))

Removed ranking rows: 2331786


In [178]:
# Re-load after fixing rankings
players = pd.read_csv("data/players_all.csv", low_memory=False)
ranks   = pd.read_csv("data/rankings_all.csv", low_memory=False)

missing_rank_players = set(ranks["player_key"]) - set(players["player_key"])
print("Missing ranking player keys in players AFTER FIX:", len(missing_rank_players))

Missing ranking player keys in players AFTER FIX: 0


In [179]:
import pandas as pd

players = pd.read_csv("data/players_all.csv")
tourns  = pd.read_csv("data/tournaments_all.csv")
matches = pd.read_csv("data/matches_all.csv")
ranks   = pd.read_csv("data/rankings_all.csv")

assert len(players) == players["player_key"].nunique()
assert len(tourns)  == tourns["tourney_key"].nunique()
assert len(matches) == matches["match_id"].nunique()
assert len(ranks)   == ranks["ranking_key"].nunique()

players_set = set(players["player_key"])
assert len(set(matches["player_winner_key"]) - players_set) == 0
assert len(set(matches["player_loser_key"])  - players_set) == 0
assert len(set(ranks["player_key"]) - players_set) == 0
print("All Neo4j integrity checks passed.")

All Neo4j integrity checks passed.
